# Session 3: how far does it sink, and how fast?
**A. Real-world problem.** A footing can be strong enough and still sink too far, or sink
more under one column than under its neighbour, which cracks walls. Settlement has two
parts: an immediate elastic part and, in clay, a slow squeezing-out of water called
consolidation that takes years. Today you compute both, check the stress spreading with
2:1 and with Boussinesq, validate against closed forms, and then size a small building's
footings for even settlement. Educational model only: never use it to design a real foundation.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
GAMMA_CONCRETE = 24.0     # kN/m^3
I_RIGID_SQUARE = 0.82     # our chosen influence factor for a rigid square footing (textbook value)

## Step 1: immediate (elastic) settlement
The soil is treated as an elastic half-space with Young's modulus E (kPa) and Poisson's
ratio nu. A footing of width B carrying pressure q settles

    s = q B (1 - nu^2) I / E

I is an influence factor that depends on the footing shape and stiffness. We choose
I = 0.82 for a rigid square footing (a textbook value; the checkpoint brackets it between
the exact rigid circle, pi/4 = 0.785, and the flexible square average, about 0.95).

In [ ]:
def immediate_settlement(q_kpa, width_m, e_kpa, nu=0.3, influence=I_RIGID_SQUARE):
    """Elastic settlement (m)."""
    return q_kpa * width_m * (1 - nu ** 2) * influence / e_kpa


def _corner(a, b):
    """Corner settlement of a uniformly loaded a x b rectangle, in units of q (1 - nu^2) / (pi E)."""
    d = math.hypot(a, b)
    return a * math.log((b + d) / a) + b * math.log((a + d) / b)


def flexible_square_factors(n=40):
    """Influence factors (centre, average) of a UNIFORMLY loaded flexible square, by superposition of corner formulas."""
    pts = (np.arange(n) + 0.5) / n          # side = 1
    vals = np.zeros((n, n))
    for i, x in enumerate(pts):
        for j, y in enumerate(pts):
            vals[i, j] = sum(_corner(u, v) for u in (x, 1 - x) for v in (y, 1 - y)) / math.pi
    return 4 * _corner(0.5, 0.5) / math.pi, float(vals.mean())

## Step 2: consolidation settlement of a clay layer
When a clay layer is loaded, water is slowly squeezed out and the layer thins. For a layer
of thickness H with compression index Cc, initial void ratio e0, initial effective stress
sigma0 at mid-layer and added stress dsigma:

    s = H Cc / (1 + e0) * log10( (sigma0 + dsigma) / sigma0 )

How fast? With time factor T_v = c_v t / H_dr^2 (c_v = coefficient of consolidation,
H_dr = longest drainage path) the average degree of consolidation is the series

    U = 1 - sum over m of  2 / M^2 * exp(-M^2 T_v),   M = (2m + 1) pi / 2.

In [ ]:
def consolidation_settlement(h_m, cc, e0, sigma0_kpa, dsigma_kpa):
    return h_m * cc / (1 + e0) * math.log10((sigma0_kpa + dsigma_kpa) / sigma0_kpa)


def degree_of_consolidation(tv, terms=2000):
    m = (np.arange(terms) + 0.5) * math.pi
    return float(1 - np.sum(2 / m ** 2 * np.exp(-m ** 2 * tv)))


def time_factor_for(u, tol=1e-12):
    """Invert U(T_v) by bisection (0 < u < 1)."""
    lo, hi = 0.0, 5.0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if degree_of_consolidation(mid) < u:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


def time_for_degree(u, cv_m2_per_year, h_dr_m):
    """Time in years to reach average degree of consolidation u."""
    return time_factor_for(u) * h_dr_m ** 2 / cv_m2_per_year


CLAY_TOP_M = 2.0         # clay starts 2 m below the footing base
FOOTING_DEPTH_M = 1.0    # footing base is 1 m below the ground surface
GAMMA_EFF = 9.0          # effective unit weight of the soil, kN/m^3 (assumed)
CC, E0 = 0.3, 0.9        # compression index and initial void ratio of the clay (assumed)


def clay_layer_response(load_kn, side_m, h_dr_m, cv_m2_per_year):
    """Double-drained clay layer of thickness 2 * h_dr below a square footing (2:1 spread to mid-layer).

    Returns (added stress at mid-layer in kPa, initial effective stress in kPa, final settlement in m, t90 in years).
    """
    z_mid = CLAY_TOP_M + h_dr_m
    dsig = stress_2to1(load_kn, side_m, z_mid)
    sig0 = GAMMA_EFF * (FOOTING_DEPTH_M + z_mid)
    s_final = consolidation_settlement(2 * h_dr_m, CC, E0, sig0, dsig)
    return dsig, sig0, s_final, time_for_degree(0.9, cv_m2_per_year, h_dr_m)

## Step 3: how the stress spreads with depth
Under a footing carrying P, the added vertical stress dies away with depth z.
* 2:1 approximation (a quick rule, only an approximation): the load spreads at a slope of
  2 vertical to 1 horizontal, so a square footing of side B loads a square of side B + z:
  dsigma = P / (B + z)^2.
* Boussinesq (elastic half-space), point load P, directly below it:
  dsigma = 3 P / (2 pi z^2). In general dsigma = 3 P z^3 / (2 pi R^5), R = distance to the load.
Add up point loads over the footing area to get the stress under a real footing.

In [ ]:
def stress_2to1(load_kn, side_m, z_m):
    return load_kn / (side_m + z_m) ** 2


def boussinesq_axis(load_kn, z_m):
    """Vertical stress (kPa) on the axis of a point load."""
    return 3 * load_kn / (2 * math.pi * z_m ** 2)


def boussinesq_point(load_kn, x_m, y_m, z_m):
    r = math.sqrt(x_m ** 2 + y_m ** 2 + z_m ** 2)
    return 3 * load_kn * z_m ** 3 / (2 * math.pi * r ** 5)


def stress_square_centre(q_kpa, side_m, z_m, n=60):
    """Stress under the centre of a uniformly loaded square, by summing n x n point loads."""
    h = side_m / n
    xs = (np.arange(n) + 0.5) * h - side_m / 2
    gx, gy = np.meshgrid(xs, xs)
    r = np.sqrt(gx ** 2 + gy ** 2 + z_m ** 2)
    return float(np.sum(3 * q_kpa * h * h * z_m ** 3 / (2 * math.pi * r ** 5)))


def stress_circle_numeric(q_kpa, radius_m, z_m, n=2000):
    """Stress under the centre of a uniformly loaded circle, by summing rings of point loads."""
    dr = radius_m / n
    r = (np.arange(n) + 0.5) * dr
    dp = q_kpa * 2 * math.pi * r * dr
    big_r = np.sqrt(r ** 2 + z_m ** 2)
    return float(np.sum(3 * dp * z_m ** 3 / (2 * math.pi * big_r ** 5)))


def stress_circle_closed(q_kpa, radius_m, z_m):
    """Closed form under the centre of a uniformly loaded circle."""
    return q_kpa * (1 - (1 / (1 + (radius_m / z_m) ** 2)) ** 1.5)

## Step 4: failure cases
1. Bearing failure: the soil shears when the pressure reaches q_ult; we require the
   pressure to stay below q_ult / FS.
2. Differential settlement: neighbouring footings settle by different amounts.

In [ ]:
def strip_bearing_check(line_load_knm, width_m, cohesion_kpa, thickness_m=0.6, fs_required=3.0):
    """Check a strip footing on phi = 0 clay. Returns (pressure, q_ult, FS, passes)."""
    q = line_load_knm / width_m + GAMMA_CONCRETE * thickness_m
    q_ult = 5.7 * cohesion_kpa
    fs = q_ult / q
    return q, q_ult, fs, fs >= fs_required


def collapse_line_load(width_m, cohesion_kpa, thickness_m=0.6):
    """Line load (kN/m) at which the pressure equals q_ult (FS = 1)."""
    return (5.7 * cohesion_kpa - GAMMA_CONCRETE * thickness_m) * width_m


def settlements(loads_kn, sides_m, e_kpa, nu=0.3, influence=I_RIGID_SQUARE):
    """Elastic settlement (m) of each square footing under its column load (net pressure P / B^2)."""
    return [immediate_settlement(p / b ** 2, b, e_kpa, nu, influence) for p, b in zip(loads_kn, sides_m)]


def differential(values):
    return max(values) - min(values)

## Step 5: mini-project helper, even settlement
Elastic settlement of a rigid square footing is s = k P / B with k = (1 - nu^2) I / E:
for the same load a wider footing sinks LESS. Bearing needs B >= B_bear; a target
settlement s_t needs B >= k P / s_t. Take the larger, rounded up to 0.05 m.

In [ ]:
def bearing_side(load_kn, q_allow_kpa, thickness_m=0.6, step=0.05):
    net = q_allow_kpa - GAMMA_CONCRETE * thickness_m
    return math.ceil(math.sqrt(load_kn / net) / step - 1e-9) * step


def design_for_target(loads_kn, q_allow_kpa, s_target_m, e_kpa, nu=0.3, influence=I_RIGID_SQUARE, step=0.05):
    """Sides (m) meeting both the bearing limit and a target settlement."""
    k = (1 - nu ** 2) * influence / e_kpa
    return [max(bearing_side(p, q_allow_kpa), math.ceil(k * p / s_target_m / step - 1e-9) * step) for p in loads_kn]

## Step 6: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("elastic: q 150 kPa, B 2 m, E 20 MPa, nu 0.3 ->", round(immediate_settlement(150, 2.0, 20000) * 1000, 1), "mm")
    print("flexible square factors (centre, average):", [round(v, 3) for v in flexible_square_factors()])
    s = consolidation_settlement(4.0, 0.3, 0.9, 100.0, 50.0)
    print("consolidation settlement:", round(s * 1000, 1), "mm")
    print("Tv for U = 50%, 90%:", round(time_factor_for(0.5), 3), round(time_factor_for(0.9), 3))
    print("t50, t90 (cv 1.5 m2/yr, Hdr 2 m):", round(time_for_degree(0.5, 1.5, 2.0), 2), round(time_for_degree(0.9, 1.5, 2.0), 2), "years")
    print("z = 3 m under 600 kN / 2 m square:", round(stress_2to1(600, 2, 3), 1), round(stress_square_centre(150, 2, 3), 1), round(boussinesq_axis(600, 3), 1))
    print("circle a = 1, z = 1:", round(stress_circle_numeric(100, 1, 1), 2), round(stress_circle_closed(100, 1, 1), 2))
    print("strip 150 kN/m on B 1.5 c 50:", [round(v, 2) if not isinstance(v, bool) else v for v in strip_bearing_check(150, 1.5, 50)], "collapse", round(collapse_line_load(1.5, 50), 1), "kN/m")

    loads = [250, 450, 800, 1200]
    std = [bearing_side(p, 150) for p in loads]
    print("standard sides", [round(b, 2) for b in std], "settlements mm", [round(v * 1000, 1) for v in settlements(loads, std, 20000)])
    for st in (0.008, 0.010, 0.012):
        ev = design_for_target(loads, 150, st, 20000)
        sv = settlements(loads, ev, 20000)
        print("target", st * 1000, "mm sides", [round(b, 2) for b in ev], "settle", [round(v * 1000, 1) for v in sv],
              "diff", round(differential(sv) * 1000, 2), "area", round(sum(b * b for b in ev), 1))

    z = np.linspace(0.5, 8, 100)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(z, [stress_2to1(600, 2, v) for v in z], label="2:1 approximation")
    ax.plot(z, [stress_square_centre(150, 2, v) for v in z], label="Boussinesq, sum over the footing")
    ax.plot(z, [boussinesq_axis(600, v) for v in z], "--", label="Boussinesq, one point load")
    ax.set_xlabel("depth below footing z (m)")
    ax.set_ylabel("added vertical stress on the axis (kPa)")
    ax.set_title("600 kN on a 2 m square footing")
    ax.set_ylim(0, 160)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "stress_with_depth.png", dpi=120)

    tv = np.logspace(-3, 1, 200)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.semilogx(tv, [100 * degree_of_consolidation(t) for t in tv], color="C0")
    ax2.plot([time_factor_for(0.5), time_factor_for(0.9)], [50, 90], "ro")
    ax2.set_xlabel("time factor T_v (no unit, log scale)")
    ax2.set_ylabel("average degree of consolidation U (%)")
    ax2.set_title("U = 50% at T_v = 0.197, U = 90% at T_v = 0.848")
    ax2.grid(alpha=0.3, which="both")
    fig2.tight_layout()
    fig2.savefig(OUT / "consolidation_curve.png", dpi=120)

    fig3, ax3 = plt.subplots(figsize=(6, 4))
    ev = design_for_target(loads, 150, 0.010, 20000)
    xs = np.arange(len(loads))
    ax3.bar(xs - 0.2, [v * 1000 for v in settlements(loads, std, 20000)], 0.4, label="sized for bearing only")
    ax3.bar(xs + 0.2, [v * 1000 for v in settlements(loads, ev, 20000)], 0.4, label="sized for 10 mm target")
    ax3.set_xticks(xs, [f"{p} kN" for p in loads])
    ax3.set_xlabel("column load")
    ax3.set_ylabel("elastic settlement (mm)")
    ax3.set_title("Differential settlement of four footings")
    ax3.grid(alpha=0.3, axis="y")
    ax3.legend()
    fig3.tight_layout()
    fig3.savefig(OUT / "differential_settlement.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `time_for_degree(0.9, cv, hdr)` to find how long 90 percent consolidation takes for c_v = 1.5 m2/year and drainage paths of 1, 2 and 4 m. Predict the ratios first.
2. **Stretch.** Run the mini-project scan with E = 10 MPa instead of 20 MPa; before running it, predict whether the best target settlement doubles.
3. **Explain.** Explain why the 2:1 spread stress at depth is smaller than the Boussinesq sum under the footing centre, and where the 2:1 rule gets it right.

## Assessment
- **A. Conceptual.** Why does consolidation take years in clay, and why does halving the drainage path make it much faster?
- **B. Numerical.** A clay layer 3 m thick (Cc 0.3, e0 0.9) has sigma0 = 100 kPa and dsigma = 50 kPa at mid-layer. Find the final consolidation settlement.
- **C. Graph and simulation interpretation.** On the consolidation curve with c_v = 1.5 m2/year and H_dr = 2 m, at what times are 50 and 90 percent of the settlement reached?
- **D. Design and optimisation challenge.** Design: two columns carry 250 kN and 1200 kN. The bearing-only footings are 1.40 m and 3.00 m (settlements 6.7 and 14.9 mm). With E = 20 MPa and nu = 0.3, find the smallest width for the big footing in 0.05 m steps so that the two settlements differ by at most 3.5 mm. What does it cost?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    s = L.immediate_settlement(150, 2.0, 20000)
    checks.append(("worked example: elastic settlement 150 * 2 * 0.91 * 0.82 / 20000 = 11.2 mm", abs(s - 150 * 2 * 0.91 * 0.82 / 20000) < 1e-12 and abs(s * 1000 - 11.19) < 0.01))
    centre, avg = L.flexible_square_factors()
    checks.append(("flexible square: centre factor 1.122 and average factor 0.95", abs(centre - 1.122) < 0.002 and abs(avg - 0.95) < 0.01))
    checks.append(("chosen I = 0.82 lies between the rigid circle pi/4 = 0.785 and the flexible square average 0.95", math.pi / 4 < L.I_RIGID_SQUARE < avg))
    checks.append(("settlement is proportional to the width at equal pressure", abs(L.immediate_settlement(100, 4, 20000) / L.immediate_settlement(100, 2, 20000) - 2.0) < 1e-12))
    sc = L.consolidation_settlement(4.0, 0.3, 0.9, 100.0, 50.0)
    checks.append(("worked example: consolidation settlement 4 * 0.3 / 1.9 * log10(1.5) = 111.2 mm",
                   abs(sc - 4 * 0.3 / 1.9 * math.log10(1.5)) < 1e-12 and abs(sc * 1000 - 111.2) < 0.05))
    checks.append(("no added stress gives no settlement", abs(L.consolidation_settlement(4.0, 0.3, 0.9, 100.0, 0.0)) < 1e-12))
    checks.append(("U = 50% at T_v = 0.197", abs(L.time_factor_for(0.5) - 0.197) < 0.001))
    checks.append(("U = 90% at T_v = 0.848", abs(L.time_factor_for(0.9) - 0.848) < 0.001))
    checks.append(("U(0) is 0 and U(10) is 1 (limits of the series)", abs(L.degree_of_consolidation(0.0)) < 1e-3 and abs(L.degree_of_consolidation(10.0) - 1.0) < 1e-9))
    tvs = np.linspace(0.0, 2.0, 50)
    us = [L.degree_of_consolidation(t) for t in tvs]
    checks.append(("U never decreases as T_v grows", all(b >= a - 1e-12 for a, b in zip(us, us[1:]))))
    checks.append(("early-time check: U = sqrt(4 T_v / pi) at T_v = 0.197 gives 0.50", abs(math.sqrt(4 * 0.197 / math.pi) - 0.5) < 0.01))
    t50, t90 = L.time_for_degree(0.5, 1.5, 2.0), L.time_for_degree(0.9, 1.5, 2.0)
    checks.append(("worked example: c_v 1.5 m2/yr, H_dr 2 m gives t50 = 0.52 years and t90 = 2.26 years", abs(t50 - 0.5249) < 0.005 and abs(t90 - 2.2615) < 0.005))
    checks.append(("doubling the drainage path makes consolidation 4 times slower", abs(L.time_for_degree(0.9, 1.5, 4.0) / t90 - 4.0) < 1e-9))
    checks.append(("hand calculation: 3 * 600 / (2 pi * 9) = 31.8 kPa on the axis of a point load at 3 m",
                   abs(L.boussinesq_axis(600, 3.0) - 31.83) < 0.01 and abs(L.boussinesq_axis(600, 3.0) - L.boussinesq_point(600, 0.0, 0.0, 3.0)) < 1e-9))
    checks.append(("2:1 spread: 600 / (2 + 3)^2 = 24.0 kPa at 3 m depth", abs(L.stress_2to1(600, 2, 3) - 24.0) < 1e-12))
    num = L.stress_square_centre(150, 2, 3)
    checks.append(("Boussinesq sum over the 2 m square at 3 m depth is 26.8 kPa, so 2:1 is about 11% low", abs(num - 26.84) < 0.05 and 0.08 < (num - 24.0) / num < 0.14))
    checks.append(("close to the footing the stress is the footing pressure (z = 0.05 m gives about 150 kPa)", abs(L.stress_square_centre(150, 2, 0.05, n=120) - 150) < 3.0))
    checks.append(("far below the footing the stress tends to the point-load value (z = 50 m within 1%)", abs(L.stress_square_centre(150, 2, 50.0) / L.boussinesq_axis(600, 50.0) - 1) < 0.01))
    a, b = L.stress_circle_numeric(100, 1.0, 1.0), L.stress_circle_closed(100, 1.0, 1.0)
    checks.append(("validation: circle numeric sum 64.64 kPa equals the closed form 64.64 kPa at depth = radius", abs(a - b) < 1e-3 and abs(b - 64.645) < 0.01))
    q, qult, fs, ok = L.strip_bearing_check(150, 1.5, 50)
    checks.append(("failure case: 150 kN/m on 1.5 m of c = 50 kPa clay gives 114.4 kPa, FS 2.49, below the required 3",
                   abs(q - 114.4) < 0.01 and abs(fs - 2.49) < 0.01 and not ok))
    checks.append(("collapse load is 405.9 kN/m and there the factor of safety is 1", abs(L.collapse_line_load(1.5, 50) - 405.9) < 0.05 and abs(L.strip_bearing_check(L.collapse_line_load(1.5, 50), 1.5, 50)[2] - 1.0) < 1e-9))
    loads = [250, 450, 800, 1200]
    std = [L.bearing_side(p, 150) for p in loads]
    ss = L.settlements(loads, std, 20000)
    checks.append(("standard layout: sides 1.40, 1.85, 2.45, 3.00 m, differential settlement 8.26 mm, plan area 20.4 m2",
                   np.allclose(std, [1.4, 1.85, 2.45, 3.0]) and abs(L.differential(ss) * 1000 - 8.26) < 0.01 and abs(sum(b * b for b in std) - 20.385) < 0.01))
    checks.append(("the heaviest column settles the most when footings are sized for bearing only", int(np.argmax(ss)) == 3))
    best = None
    for k in range(12, 41):
        sides = L.design_for_target(loads, 150, k * 0.5e-3, 20000)
        sv = L.settlements(loads, sides, 20000)
        area = sum(b * b for b in sides)
        if L.differential(sv) <= 0.004 and (best is None or area < best[0]):
            best = (area, k * 0.5, sides, L.differential(sv))
    checks.append(("mini-project: smallest area with differential <= 4 mm is target 10.5 mm, sides 1.40, 1.85, 2.85, 4.30 m, 32.0 m2, 3.81 mm",
                   best is not None and abs(best[1] - 10.5) < 1e-9 and np.allclose(best[2], [1.4, 1.85, 2.85, 4.3]) and abs(best[0] - 32.0) < 0.05 and abs(best[3] * 1000 - 3.81) < 0.01))
    checks.append(("every mini-project footing still meets bearing (pressure with self-weight <= 150 kPa)",
                   all(p / b ** 2 + 24 * 0.6 <= 150 + 1e-9 for p, b in zip(loads, best[2]))))
    checks.append(("worked example: total settlement 11.2 + 111.2 = 122.4 mm", abs((s + sc) * 1000 - 122.4) < 0.05))
    d0, sg0, sf0, t90a = L.clay_layer_response(600, 2.0, 2.0, 1.5)
    checks.append(("app default case: added stress 16.7 kPa on 45 kPa, final settlement 86.4 mm, t90 2.26 years",
                   abs(d0 - 16.667) < 0.01 and abs(sg0 - 45.0) < 1e-9 and abs(sf0 * 1000 - 86.42) < 0.01 and abs(t90a - 2.26) < 0.01))
    d1, sg1, sf1, t901 = L.clay_layer_response(600, 4.0, 1.5, 3.0)
    checks.append(("app challenge: 600 kN, B 4 m, H_dr 1.5 m, c_v 3 gives 48.1 mm and 0.64 years (meets 60 mm and 1 year)",
                   abs(sf1 * 1000 - 48.09) < 0.02 and abs(t901 - 0.636) < 0.005 and sf1 <= 0.06 and t901 <= 1.0))
    checks.append(("shipped one-shot consolidation at 2 years for the default layer is 75.4 mm",
                   abs(sf0 * L.degree_of_consolidation(1.5 * 2.0 / 4.0) * 1000 - 75.4) < 0.1))
    checks.append(("standard layout settlements are 6.7, 9.1, 12.2 and 14.9 mm", np.allclose(np.array(ss) * 1000, [6.66, 9.1, 12.2, 14.9], atol=0.06)))
    ds = L.settlements([250, 1200], [1.4, 4.45], 20000)
    ds4 = L.settlements([250, 1200], [1.4, 4.4], 20000)
    checks.append(("assessment D: B = 4.40 m gives a 3.51 mm difference (too much), 4.45 m gives 3.4 mm, 4.406 m is the exact limit and the area grows from 9.0 to 19.8 m2",
                   abs(L.differential(ds4) * 1000 - 3.51) < 0.01 and L.differential(ds4) > 0.0035 and abs(L.differential(ds) * 1000 - 3.40) < 0.01
                   and abs(1200 * 0.91 * 0.82 / 20000 / (0.0035 + 250 * 0.91 * 0.82 / 20000 / 1.4) - 4.4056) < 0.001 and abs(4.45 ** 2 - 19.80) < 0.01))
    checks.append(("a 4 m thick layer settles 4/3 as much as a 3 m layer (assessment B: 83.4 mm)", abs(L.consolidation_settlement(3.0, 0.3, 0.9, 100.0, 50.0) * 1000 - 83.4) < 0.05))
    checks.append(("half the consolidation settlement comes in about 6 months for the worked layer (0.52 years)", abs(t50 * 12 - 6.3) < 0.1))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
